In [ ]:
import torch 
import torch.nn.functional as F
import matplotlib.pyplot as plt
from torch.utils.data import TensorDataset, random_split, DataLoader

In [ ]:
words = open('names.txt', 'r').read().splitlines()

chars = ['.'] + sorted(list(set(''.join(words))))

stoi = {char : i for i, char in zip(range(len(chars)), chars)}
itost = {i : char for i, char in zip(range(len(chars)), chars)}

vocab_size = len(chars)

X, Y = [], []
block_size = 8

for w in words:
    context = [0] * block_size

    for ch in w + '.':
        idx = stoi[ch]
        X.append(context)
        Y.append(idx)
        context = context[1:] + [idx] 

X = torch.tensor(X)
Y = torch.tensor(Y)

    
dataset = TensorDataset(X, Y)

total_size = len(dataset)
train_size = int(0.8 * total_size)
val_size = int(0.1 * total_size)
test_size = total_size - train_size - val_size

g = torch.Generator().manual_seed(42)

train_set, val_set, test_set = random_split(dataset, [train_size, val_size, test_size], generator=g)

X_train, Y_train = X[train_set.indices], Y[train_set.indices]
X_val,   Y_val  = X[val_set.indices],   Y[val_set.indices]
X_test,  Y_test  = X[test_set.indices],  Y[test_set.indices]

print(len(train_set))



In [ ]:
n_embd = 20 
n_hidden = 300 

g = torch.Generator().manual_seed(42) 
C  = torch.randn((vocab_size, n_embd), generator=g)

W1 = torch.randn((n_embd * block_size, n_hidden), generator=g) * (5/3)/((n_embd * block_size)**0.5)
b1 = torch.randn(n_hidden,                        generator=g) * 0.1

W2 = torch.randn((n_hidden, vocab_size),          generator=g) * 0.1
b2 = torch.randn(vocab_size,                      generator=g) * 0.1

bngain = torch.randn((1, n_hidden))*0.1 + 1.0
bnbias = torch.randn((1, n_hidden))*0.1

parameters = [C, W1, b1, W2, b2, bngain, bnbias]

for p in parameters:
  p.requires_grad = True

max_steps = 200000
batch_size = 32
n = batch_size 
lossi = []

for i in range(max_steps):

  # forward
  ix = torch.randint(0, X_train.shape[0], (batch_size,), generator=g)
  Xb, Yb = X_train[ix], Y_train[ix] 

  emb = C[Xb] 
  
  embcat = emb.view(emb.shape[0], -1) 
  
  hprebn = embcat @ W1 + b1 
  bnmean = hprebn.mean(0, keepdim=True)
  bndiff = hprebn - bnmean 
  bndiff2 = bndiff**2     
  bnvar = hprebn.var(0, keepdim=True, unbiased=True)
  bnvar_inv = (bnvar + 1e-5)**-0.5
  bnraw = bndiff * bnvar_inv
  hpreact = bngain * bnraw + bnbias
  
  
  h = torch.tanh(hpreact) 
  logits = h @ W2 + b2 
  loss = F.cross_entropy(logits, Yb)

  # manual backprop

  dC, dW1, db1, dW2, db2, dbngain, dbnbias = None, None, None, None, None, None, None

  dlogits = F.softmax(logits, 1)
  dlogits[range(n), Yb] -= 1
  dlogits /= n
  dh = dlogits @ W2.T
  dW2 = h.T @ dlogits
  db2 = dlogits.sum(0) 

  
  dhpreact = dh * (1.0 - h**2)
  
  dbngain = (dhpreact * bnraw).sum(0, keepdim=True)
  dbnraw = dhpreact * bngain
  dbnbias = dhpreact.sum(0, keepdim=True)
  dbnvar_inv = (dbnraw * bndiff).sum(0, keepdim=True)
  dbnvar = dbnvar_inv * (-0.5*(bnvar + 1e-5)**-1.5)
  dbndiff2 = dbnvar * (1/(n-1))*torch.ones_like(bndiff2)
  dbndiff = dbndiff2 * 2*bndiff + dbnraw * bnvar_inv
  dbnmeani = -dbndiff.sum(0, keepdim=True)
  dhprebn = dbnmeani * (1/n)*torch.ones_like(hprebn) + dbndiff
  
  dembcat = dhprebn @ W1.T
  dW1 = embcat.T @ dhprebn
  db1 = dhprebn.sum(0) 
  
  demb = dembcat.view(emb.shape)
  dC = torch.zeros_like(C)
  dC.index_add_(0, Xb.view(-1), demb.view(-1, n_embd))

  manual_grads = [dC, dW1, db1, dW2, db2, dbngain, dbnbias]

  lr = 0.1 if i < 100000 else 0.01 
  for p, grad in zip(parameters, manual_grads):
    p.data += -lr * grad 

  
  if i % 10000 == 0: 
    print(f'{i:7d}/{max_steps:7d}: {loss.item():.4f}')
  lossi.append(loss.log10().item())




In [ ]:
# Préparation à l'inférence / Calibration batchnorm 

with torch.no_grad():
     emb = C[X_train]
     embcat = emb.view(emb.shape[0], -1)
     hprebn = embcat @ W1 + b1
     bnmean = hprebn.mean(0, keepdim=True)
     bndiff = hprebn - bnmean 
     bndiff2 = bndiff**2     
     bnvar = hprebn.var(0, keepdim=True, unbiased=True)
     bnvar_inv = (bnvar + 1e-5)**-0.5

     

In [ ]:
# Inférence 

@torch.no_grad()

def split_loss (split):
    x, y = {
    'train': (X_train, Y_train),
    'val': (X_val, Y_val),
    'test': (X_test, Y_test),
    }[split]
    emb = C[x]
    embcat = emb.view(emb.shape[0], -1)
    hprebn = embcat @ W1 + b1
    hpreact = bngain * (hprebn - bnmean) * bnvar_inv + bnbias
    h = torch.tanh(hpreact)
    logits = h @ W2 + b2
    loss = F.cross_entropy(logits, y)
    return split, loss.item()

print(split_loss('train'),
split_loss('val'),
split_loss('test'))

In [ ]:

for i in range (20):
    ch = '.'
    context = [0] * block_size
    w = []
    while True: 
        emb = C[torch.tensor(context)]
        embcat = emb.view(-1, n_embd * block_size)
        hprebn = embcat @ W1 + b1
        hpreact = bngain * (hprebn - bnmean) * bnvar_inv + bnbias
        h = torch.tanh(hpreact)
        logits = h @ W2 + b2
        probs = torch.softmax(logits, dim=-1)
        id = torch.multinomial(probs, num_samples=1)
        ch = itost[id.item()]
        context = context[1:] + [id.item()]
        
        if ch != '.':
            w.append(ch)
        else : 
            break
    
    print(''.join(w))

